---
image: example.gif
pub-info:
    abstract: |
        A more realistic pathway, where entities can branch down different routes and pass through
        multiple queueing and resource-use steps. Also demonstrates 'setup_mode', which overlays a
        coordinate grid on your background image to help you place resources and queues accurately.
execute: 
  enabled: true
---

# Animation of a SimPy Model with Branching and Multiple Steps

In [ ]:
#| echo: false
import plotly.io as pio
pio.renderers.default = "notebook"
import os

In [ ]:
from ex_2_model_classes import Trial, g

from vidigi.animation import animate_activity_log
from vidigi.utils import EventPosition, create_event_position_df

In [ ]:
#| echo: false
#| output: asis
# Path to the external Python script
file_path = "ex_2_model_classes.py"

# Read the file content
if os.path.exists(file_path):
    with open(file_path, "r") as f:
        code_content = f.read()
else:
    code_content = "File not found."
with open(file_path, "r") as f:
    code_content = f.read()

# Print the Quarto `{details}` block for collapsible output
print(f"""
:::{{.callout-note collapse="true"}}
### View Imported Code, which has had logging steps added at the appropriate points in the 'model' class

```python
{code_content}
```

:::

""")

In [ ]:
my_trial = Trial()

my_trial.run_trial()

Let's take a look at some sample logs from a single run. 

In [ ]:
my_trial.trial_logger.get_log_by_run(run=1, as_df=True).head(20)

In [ ]:
event_position_df = create_event_position_df(
    [
        EventPosition(event="arrival", x=10, y=250, label="Arrival"),
        # Triage - minor and trauma
        EventPosition(
            event="triage_wait_begins", x=160, y=375, label="Waiting for<br>Triage"
        ),
        EventPosition(
            event='triage_start',
            x=160,
            y=315,
            resource="n_triage",
            label="Being Triaged",
        ),
        # Minors (non-trauma) pathway
        EventPosition(
            event="MINORS_registration_wait_begins",
            x=300,
            y=145,
            label="Waiting for<br>Registration",
        ),
        EventPosition(
            event="registration_start",
            x=300,
            y=85,
            resource="n_reg",
            label="Being<br>Registered",
        ),
        EventPosition(
            event="MINORS_examination_wait_begins",
            x=465,
            y=145,
            label="Waiting for<br>Examination",
        ),
        EventPosition(
            event='exam_start',
            x=465,
            y=85,
            resource="n_exam",
            label="Being<br>Examined",
        ),
        EventPosition(
            event="MINORS_treatment_wait_begins",
            x=630,
            y=145,
            label="Waiting for<br>Treatment",
        ),
        EventPosition(
            event='non_trauma_treatment_start',
            x=630,
            y=85,
            resource="n_cubicles_non_trauma_treat",
            label="Being<br>Treated",
        ),
        # Trauma pathway
        EventPosition(
            event="TRAUMA_stabilisation_wait_begins",
            x=300,
            y=560,
            label="Waiting for<br>Stabilisation",
        ),
        EventPosition(
            event='trauma_stabilisation_start',
            x=300,
            y=490,
            resource="n_trauma",
            label="Being<br>Stabilised",
        ),
        EventPosition(
            event="TRAUMA_treatment_wait_begins",
            x=630,
            y=560,
            label="Waiting for<br>Treatment",
        ),
        EventPosition(
            event='trauma_treatment_start',
            x=630,
            y=490,
            resource="n_cubicles_trauma_treat",
            label="Being<br>Treated",
        ),
        EventPosition(event="depart", x=670, y=330, label="Exit"),
    ]
)

## setup_mode = True

setup_mode allows us to see how the coordinates of our plot relate to the positioning of our background image, allowing us to more accurately place our entities.

In [ ]:
animate_activity_log(
    event_log=my_trial.trial_logger,
    run_number=1,
    event_position_df=event_position_df,
    scenario=g(),
    debug_mode=True,
    setup_mode=True,
    every_x_time_units=5,
    include_play_button=True,
    gap_between_entities=11,
    gap_between_resources=15,
    gap_between_resource_rows=30,
    gap_between_queue_rows=30,
    plotly_height=600,
    plotly_width=1000,
    override_x_max=700,
    override_y_max=675,
    entity_icon_size=10,
    resource_icon_size=13,
    text_size=15,
    wrap_queues_at=10,
    step_snapshot_max=20,
    limit_duration=g.sim_duration,
    time_display_units="dhm",
    display_stage_labels=False,
    add_background_image="https://raw.githubusercontent.com/Bergam0t/vidigi/refs/heads/main/examples/example_2_branching_multistep/Full%20Model%20Background%20Image%20-%20Horizontal%20Layout.drawio.png",
)

## setup_mode = False

We can then rerun our plot with `setup_mode=False` to remove the grid lines and axis tick marks.

In [ ]:
animate_activity_log(
    event_log=my_trial.trial_logger,
    run_number=1,
    event_position_df=event_position_df,
    scenario=g(),
    debug_mode=True,
    setup_mode=False,
    every_x_time_units=5,
    include_play_button=True,
    gap_between_entities=11,
    gap_between_resources=15,
    gap_between_resource_rows=30,
    gap_between_queue_rows=30,
    plotly_height=600,
    plotly_width=1000,
    override_x_max=700,
    override_y_max=675,
    entity_icon_size=10,
    resource_icon_size=13,
    text_size=15,
    wrap_queues_at=10,
    step_snapshot_max=20,
    limit_duration=g.sim_duration,
    time_display_units="dhm",
    display_stage_labels=False,
    add_background_image="https://raw.githubusercontent.com/Bergam0t/vidigi/refs/heads/main/examples/example_2_branching_multistep/Full%20Model%20Background%20Image%20-%20Horizontal%20Layout.drawio.png",
)

## Visualising long queues

Finally, let's rerun this with some build-up of queues by forcing faster arrivals and reducing the number of resources.

In [ ]:
g.arrival_df = "ed_arrivals_more_frequent.csv"
g.n_cubicles_trauma_treat = 3
g.n_cubicles_non_trauma_treat = 2
g.n_exam = 2

my_trial = Trial()

my_trial.run_trial()

In [ ]:
animate_activity_log(
    event_log=my_trial,
    event_position_df=event_position_df,
    run_number=1,
    scenario=g(),
    debug_mode=True,
    setup_mode=False,
    every_x_time_units=5,
    include_play_button=True,
    gap_between_entities=11,
    gap_between_resources=15,
    gap_between_resource_rows=30,
    gap_between_queue_rows=30,
    plotly_height=600,
    plotly_width=1000,
    override_x_max=700,
    override_y_max=675,
    entity_icon_size=10,
    resource_icon_size=13,
    text_size=15,
    wrap_queues_at=10,
    step_snapshot_max=20,
    limit_duration=g.sim_duration,
    time_display_units="dhm",
    display_stage_labels=False,
    add_background_image="https://raw.githubusercontent.com/Bergam0t/vidigi/refs/heads/main/examples/example_2_branching_multistep/Full%20Model%20Background%20Image%20-%20Horizontal%20Layout.drawio.png",
    step_snapshot_limit_gauges=True,
)